[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/5_Ghost_of_Machine/code/grading/evaluate.ipynb)

# Ghost of the Machine — evaluate a solution on Google Colab

Runs a solution the way the contest graded it, against the hidden leaderboard splits (now public in the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-ghost-of-the-machine)).

**Evaluating your own notebook:** upload it via the Files pane as `/content/solution.ipynb`, then Run all. Without an upload, the stock baseline is evaluated.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-ghost-of-the-machine", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)
# merged dataset root: split folders + *_answers files, public and private together
DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)
print("dataset root:", DSROOT, "->", sorted(p.name for p in DSROOT.iterdir()))


In [ ]:
EVAL_SPLIT = "test_leaderboard_a"   # or "test_leaderboard_b"

REPO = Path("/content/IOAI-2026")
if not REPO.exists(): sh(f"git clone --depth 1 https://github.com/IOAI-official/IOAI-2026 {REPO}")
CI_ROOT = REPO/"Individual-Contest/5_Ghost_of_Machine/code/grading-original"
WORK = CI_ROOT/"submission"; WORK.mkdir(exist_ok=True)

SOL = Path("/content/solution.ipynb")
if not SOL.exists(): SOL = REPO/"Individual-Contest/5_Ghost_of_Machine/code/baseline-original/solution.ipynb"
shutil.copy(SOL, WORK/"solution.ipynb"); print("evaluating:", SOL)

link(DSROOT/"train"/"data.jsonl", WORK/"dataset"/"train"/"data.jsonl")
link(DSROOT/"train_answers.jsonl", WORK/"dataset"/"train"/"answers.jsonl")
link(DSROOT/EVAL_SPLIT/"data.jsonl", WORK/"dataset"/"test_public"/"data.jsonl")

sh(f"cd {WORK} && jupyter nbconvert --to notebook --execute --output executed.ipynb "
   f"--ExecutePreprocessor.timeout=-1 solution.ipynb")

os.environ.update(NOTEBOOK_IMAGE="unused", DATASET_ROOT=str(DSROOT),
                  GRADE_SPLIT=EVAL_SPLIT, REPO_NAME="submission")
import importlib.util
spec = importlib.util.spec_from_file_location("checker", CI_ROOT/"checker.py")
checker = importlib.util.module_from_spec(spec); spec.loader.exec_module(checker)
checker.score_submission()
for rep in CI_ROOT.rglob("report.json"):
    print(rep.read_text())
